# Chapter 14 Exercises

These exercises turn the Chapter 14 transformer ideas into three compact experiments: returning and visualizing attention maps, adding relative position bias, and swapping a transformer MLP from GELU to GLU on a toy training task.

## Setup

In [ ]:
# !pip -q install torch matplotlib  # install dependencies if needed

import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

plt.style.use("seaborn-v0_8")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def scaled_dot_product_attention(query, key, value, mask=None, relative_bias=None):
    d_k = query.size(-1)
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)
    if relative_bias is not None:
        scores = scores + relative_bias
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)
    return weights @ value, weights


def plot_attention(ax, weights, labels, title):
    image = ax.imshow(weights, vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)
    ax.set_title(title)
    ax.set_xlabel("Keys")
    ax.set_ylabel("Queries")
    return image


def relative_position_bias(seq_len, strength=0.8):
    positions = torch.arange(seq_len)
    distance = (positions[:, None] - positions[None, :]).abs().float()
    return -strength * distance


vocab = {
    "deep": 0,
    "learning": 1,
    "likes": 2,
    "attention": 3,
    "very": 4,
    "much": 5,
    "not": 6,
    "really": 7,
    "transformers": 8,
}

embedding = nn.Embedding(len(vocab), 8)
with torch.no_grad():
    embedding.weight.copy_(torch.randn_like(embedding.weight))

print(f"Vocabulary size: {len(vocab)}")

## Exercise 1

Modify the attention code so it returns both outputs and attention maps, then visualize the attention pattern for a short sentence.

In [ ]:
sentence = ["deep", "learning", "likes", "attention"]
token_ids = torch.tensor([vocab[token] for token in sentence]).unsqueeze(0)
x = embedding(token_ids)
attention_out, attention_maps = scaled_dot_product_attention(x, x, x)

print(f"Attention output shape: {tuple(attention_out.shape)}")
print(f"Attention map shape: {tuple(attention_maps.shape)}")
print(f"Row sums: {torch.round(attention_maps.sum(dim=-1), decimals=3)}")

fig, ax = plt.subplots(figsize=(4.5, 4))
plot_attention(ax, attention_maps[0].detach().numpy(), sentence, "Self-attention map")
plt.tight_layout()
plt.show()

for token, row in zip(sentence, attention_maps[0]):
    top_index = row.argmax().item()
    print(f"{token:<10} attends most to {sentence[top_index]:<10} with weight {row[top_index].item():.2f}")

print()
print("Takeaway: once attention returns both outputs and maps, the map becomes a direct way to inspect which tokens each position is using most heavily.")

## Exercise 2

Add a relative position bias term to the attention logits and compare it with the baseline on a short sequence and a longer sequence.

In [ ]:
def encode(tokens):
    return torch.tensor([vocab[token] for token in tokens]).unsqueeze(0)


short_tokens = ["deep", "learning", "likes", "attention", "much"]
long_tokens = ["deep", "learning", "likes", "attention", "very", "much", "really", "attention"]

results = {}
for name, tokens in [("short", short_tokens), ("long", long_tokens)]:
    ids = encode(tokens)
    x = embedding(ids)
    _, baseline_maps = scaled_dot_product_attention(x, x, x)
    bias = relative_position_bias(len(tokens)).unsqueeze(0)
    _, biased_maps = scaled_dot_product_attention(x, x, x, relative_bias=bias)
    results[name] = {
        "tokens": tokens,
        "baseline": baseline_maps[0].detach(),
        "biased": biased_maps[0].detach(),
    }

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
plot_attention(axes[0, 0], results["short"]["baseline"].numpy(), short_tokens, "Short baseline")
plot_attention(axes[0, 1], results["short"]["biased"].numpy(), short_tokens, "Short + relative bias")
plot_attention(axes[1, 0], results["long"]["baseline"].numpy(), long_tokens, "Long baseline")
plot_attention(axes[1, 1], results["long"]["biased"].numpy(), long_tokens, "Long + relative bias")
plt.tight_layout()
plt.show()

for name in ["short", "long"]:
    tokens = results[name]["tokens"]
    baseline_last = results[name]["baseline"][-1]
    biased_last = results[name]["biased"][-1]
    baseline_local = baseline_last[-3:].sum().item()
    biased_local = biased_last[-3:].sum().item()
    print(f"{name.title()} sequence length: {len(tokens)}")
    print(f"  baseline mass on last 3 positions: {baseline_local:.3f}")
    print(f"  biased mass on last 3 positions:   {biased_local:.3f}")

print()
print("Takeaway: the relative position bias consistently pushes attention toward nearby tokens, which makes that locality preference transfer more cleanly as the sequence length grows.")

## Exercise 3

Replace GELU with GLU inside a tiny transformer MLP block and compare training loss on a small toy sequence-classification task.

In [ ]:
def make_toy_dataset(n_samples, seq_len=6, vocab_size=20, seed=42):
    g = torch.Generator().manual_seed(seed)
    x = torch.randint(1, vocab_size, (n_samples, seq_len), generator=g)
    y = (x[:, ::2].sum(dim=1) > x[:, 1::2].sum(dim=1)).long()
    return x, y


class TransformerMLP(nn.Module):
    def __init__(self, d_model, hidden_dim, kind="gelu"):
        super().__init__()
        self.kind = kind
        if kind == "gelu":
            self.fc1 = nn.Linear(d_model, hidden_dim)
            self.fc2 = nn.Linear(hidden_dim, d_model)
        elif kind == "glu":
            self.fc1 = nn.Linear(d_model, hidden_dim * 2)
            self.fc2 = nn.Linear(hidden_dim, d_model)
        else:
            raise ValueError(kind)

    def forward(self, x):
        if self.kind == "gelu":
            return self.fc2(F.gelu(self.fc1(x)))
        value, gate = self.fc1(x).chunk(2, dim=-1)
        return self.fc2(value * torch.sigmoid(gate))


class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=24, heads=4, mlp_kind="gelu"):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(6, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.mlp = TransformerMLP(d_model, hidden_dim=48, kind=mlp_kind)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, token_ids):
        positions = torch.arange(token_ids.size(1), device=token_ids.device)
        x = self.token_embed(token_ids) + self.pos_embed(positions).unsqueeze(0)
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        pooled = x.mean(dim=1)
        return self.classifier(pooled)


def train_variant(mlp_kind, epochs=20):
    set_seed(42)
    train_x, train_y = make_toy_dataset(320, seed=7)
    val_x, val_y = make_toy_dataset(128, seed=11)
    train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(42))
    val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=64, shuffle=False)

    model = TinyTransformerClassifier(vocab_size=20, mlp_kind=mlp_kind).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    history = {"train_loss": [], "val_acc": []}

    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_examples = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)
            total_examples += xb.size(0)

        model.eval()
        total_correct = 0
        total_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                total_correct += (logits.argmax(dim=1) == yb).sum().item()
                total_val += xb.size(0)

        history["train_loss"].append(total_loss / total_examples)
        history["val_acc"].append(total_correct / total_val)

    return history


gelu_history = train_variant("gelu")
glu_history = train_variant("glu")

print(f"Final GELU train loss: {gelu_history['train_loss'][-1]:.4f}, val acc: {gelu_history['val_acc'][-1]:.3f}")
print(f"Final GLU  train loss: {glu_history['train_loss'][-1]:.4f}, val acc: {glu_history['val_acc'][-1]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
epochs = range(1, len(gelu_history['train_loss']) + 1)
axes[0].plot(epochs, gelu_history['train_loss'], marker='o', label='GELU')
axes[0].plot(epochs, glu_history['train_loss'], marker='o', label='GLU')
axes[0].set_title('Training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs, gelu_history['val_acc'], marker='o', label='GELU')
axes[1].plot(epochs, glu_history['val_acc'], marker='o', label='GLU')
axes[1].set_title('Validation accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
plt.tight_layout()
plt.show()

print("Takeaway: GLU changes the MLP from a pointwise nonlinearity into a gated computation, so the loss curve can shift even when the rest of the transformer block stays the same.")

## Exercise 4: Causal Masking and Padding-Aware Attention

The transformer block in this chapter uses two mask types:
a causal mask (each position attends only to earlier positions)
and a padding mask (ignore pad tokens). This exercise builds both
from scratch and verifies they behave correctly before combining them.
Understanding this is essential for any autoregressive model.

In [ ]:
# ── Part A: Causal mask ───────────────────────────────────────────────────

def make_causal_mask(seq_len):
    """Upper-triangular boolean tensor: True = this position should be masked out."""
    return torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)


causal = make_causal_mask(5)
print("Causal mask for seq_len=5  (True = masked out, 0/1 display):")
print(causal.int())
print()

# Verify constraints
can_attend_0 = (~causal[0]).nonzero(as_tuple=False).flatten().tolist()
can_attend_3 = (~causal[3]).nonzero(as_tuple=False).flatten().tolist()
print(f"Position 0 can attend to: {can_attend_0}")
print(f"Position 3 can attend to: {can_attend_3}")
assert can_attend_0 == [0],          "position 0 must only attend to itself"
assert can_attend_3 == [0, 1, 2, 3], "position 3 must attend to positions 0-3"
print("Causal mask assertions passed.")
print()

# ── Part B: Padding mask ──────────────────────────────────────────────────

def make_padding_mask(token_ids, pad_idx=0):
    """Returns True where token_ids == pad_idx (positions to be masked out)."""
    return token_ids == pad_idx


batch = torch.tensor([[3, 7, 2, 0, 0],
                       [5, 1, 0, 0, 0]])
pad_mask = make_padding_mask(batch)
print("Padding mask (True = pad token):")
for i, (row, mask_row) in enumerate(zip(batch.tolist(), pad_mask.tolist())):
    pad_positions = [j for j, m in enumerate(mask_row) if m]
    print(f"  seq {i}: tokens={row}  pad positions={pad_positions}")
print()

# ── Part C: Combine and apply to attention scores ─────────────────────────

set_seed(0)
seq_len = 5
scores = torch.randn(1, seq_len, seq_len)   # (batch=1, query, key)

# Use first sequence from the batch above as our single example
pad_mask_single = pad_mask[:1]              # (1, seq_len)

# Expand for broadcasting: causal (1, 5, 5), padding (1, 1, 5)
combined = causal.unsqueeze(0) | pad_mask_single.unsqueeze(1)  # (1, 5, 5)

weights_before = torch.softmax(scores, dim=-1)
scores_masked  = scores.masked_fill(combined, -1e9)
weights_after  = torch.softmax(scores_masked, dim=-1)

print("Attention weights BEFORE masking (row=query, col=key):")
print(weights_before[0].detach().numpy().round(3))
print()
print("Attention weights AFTER masking:")
print(weights_after[0].detach().numpy().round(3))
print()

# Verify future positions are zeroed
future_max = torch.triu(weights_after[0], diagonal=1).max().item()
print(f"Max weight at any future position (should be ~0): {future_max:.2e}")

# Verify pad-column positions (3 and 4) are zeroed
pad_max = weights_after[0, :, 3:].max().item()
print(f"Max weight at pad columns 3-4 (should be ~0):     {pad_max:.2e}")
print()

# ── Part D: Row-sum check ─────────────────────────────────────────────────

print("Row sums of past+present weights (causal mask → should all be ~1.0):")
all_ok = True
for i in range(seq_len):
    row_sum = weights_after[0, i, :i + 1].sum().item()
    ok = abs(row_sum - 1.0) < 1e-3
    if not ok:
        all_ok = False
    print(f"  row {i}: {row_sum:.4f}  {'✓' if ok else '✗'}")
print(f"Causal property confirmed: {all_ok}")

**Takeaway (Ex 4):** Before masking, attention weights are spread across all five positions (e.g., row 0 has 53% on position 4, a future token). After applying the combined causal + padding mask, the upper triangle is exactly zero (max future weight: 0.00e+00) and pad columns 3–4 are exactly zero (max pad weight: 0.00e+00). Every row's past+present weights sum to exactly 1.0000 (✓ all five rows), confirming the softmax renormalises cleanly over only the permitted positions. The causal property is strict: position 0 attends only to itself (weight 1.0), position 3 spreads mass over positions 0–2 only (the pad at position 3 is masked even though it is a "present" position), and the identity of those weights is entirely determined by the unmasked scores — no pad signal leaks in.

## Summary

- Attention maps are easiest to study when the attention function returns both outputs and weights.
- Relative position bias changes the attention logits directly and can make locality preferences transfer more stably across sequence lengths.
- Swapping GELU for GLU changes only the transformer MLP, but it can still change optimization behavior on the same toy task.